In [1]:
# ============================================================
# ACRS LAYER 3 — Agentic Credit Risk Supervisor
# Three autonomous agents that work together:
#   Agent 1: Scoring Agent  — predict + explain each loan
#   Agent 2: Drift Agent    — detect model/data drift in real time
#   Agent 3: Compliance Agent — HITL trigger + audit trail
# ============================================================

import numpy as np
import pandas as pd
import pickle, shap, warnings
warnings.filterwarnings('ignore')

# Load data
X_train = pd.read_parquet('acrs_data/X_train.parquet')
X_val   = pd.read_parquet('acrs_data/X_val.parquet')
X_test  = pd.read_parquet('acrs_data/X_test.parquet')
y_test  = np.load('acrs_data/y_test.npy')

# Load models
with open('acrs_data/xgb_final.pkl','rb')  as f: xgb_final  = pickle.load(f)
with open('acrs_data/lgbm_final.pkl','rb') as f: lgbm_final = pickle.load(f)
with open('acrs_data/rf_final.pkl','rb')   as f: rf_final   = pickle.load(f)

# Ensemble probability function (same soft-voting as Layer 1 & 2)
def ensemble_predict(X):
    return (xgb_final.predict_proba(X)[:,1] +
            lgbm_final.predict_proba(X)[:,1] +
            rf_final.predict_proba(X)[:,1]) / 3

print(f"Loaded successfully! | X_test: {X_test.shape} | Models: XGBoost, LightGBM, RF ready")

Loaded successfully! | X_test: (504194, 83) | Models: XGBoost, LightGBM, RF ready


In [3]:
# ============================================================
# AGENT 1: SCORING AGENT
# Role: Given a loan application, this agent:
#   1. Computes the ensemble risk score (probability of default)
#   2. Assigns a risk label (Low / Medium / High)
#   3. Runs real-time SHAP — explains WHY that score was given
#   4. Returns the top 3 features driving the decision
#
# This is "real-time" SHAP: one loan at a time, not a batch.
# In Layer 2 we explained 2000 loans together (batch).
# Here Agent 1 explains each loan individually as it arrives.
# ============================================================

class ScoringAgent:
    def __init__(self, xgb_model, lgbm_model, rf_model, X_train_ref):
        self.xgb  = xgb_model
        self.lgbm = lgbm_model
        self.rf   = rf_model
        # TreeExplainer uses training data as background reference
        # It answers: "compared to an average loan, why is THIS one risky?"
        self.explainer = shap.TreeExplainer(xgb_model)

    def score(self, loan_row):
        # loan_row = single row DataFrame (1 loan application)
        prob = (self.xgb.predict_proba(loan_row)[:,1] +
                self.lgbm.predict_proba(loan_row)[:,1] +
                self.rf.predict_proba(loan_row)[:,1]) / 3
        score = float(prob[0])

        # Risk label thresholds (standard credit risk practice)
        if score < 0.30:   label = "Low Risk"
        elif score < 0.50: label = "Medium Risk"
        else:              label = "High Risk"

        # Real-time SHAP: explain this single loan
        shap_vals = self.explainer.shap_values(loan_row)[0]  # shape: (83,)

        # Top 3 features pushing score UP (positive SHAP = toward default)
        feat_names = loan_row.columns.tolist()
        shap_df = pd.DataFrame({'feature': feat_names, 'shap': shap_vals})
        top3 = shap_df.reindex(shap_df['shap'].abs().nlargest(3).index)

        return {
            'score':      round(score, 4),
            'label':      label,
            'top3_shap':  top3[['feature','shap']].values.tolist()
        }

# Initialise Agent 1
agent1 = ScoringAgent(xgb_final, lgbm_final, rf_final, X_train)
print("Agent 1 (Scoring) initialised")

# ── Test: score 3 individual loans ──────────────────────────
print("\n--- Agent 1 Demo: Scoring 3 loans ---")
for i in [0, 1, 2]:
    loan = X_test.iloc[[i]]          # single row, kept as DataFrame
    result = agent1.score(loan)
    print(f"\nLoan #{i}  |  True label: {'DEFAULT' if y_test[i]==1 else 'PAID'}")
    print(f"  Score : {result['score']}  →  {result['label']}")
    print(f"  Top reasons:")
    for feat, val in result['top3_shap']:
        direction = "↑ risk" if val > 0 else "↓ risk"
        print(f"    {feat:<35} SHAP={val:+.4f}  {direction}")

Agent 1 (Scoring) initialised

--- Agent 1 Demo: Scoring 3 loans ---

Loan #0  |  True label: PAID
  Score : 0.2031  →  Low Risk
  Top reasons:
    sub_grade                           SHAP=+0.2915  ↑ risk
    term                                SHAP=-0.1976  ↓ risk
    annual_inc                          SHAP=-0.1858  ↓ risk

Loan #1  |  True label: PAID
  Score : 0.342  →  Medium Risk
  Top reasons:
    dti                                 SHAP=+0.3435  ↑ risk
    term                                SHAP=+0.3149  ↑ risk
    sub_grade                           SHAP=+0.1704  ↑ risk

Loan #2  |  True label: PAID
  Score : 0.0315  →  Low Risk
  Top reasons:
    sub_grade                           SHAP=-0.5398  ↓ risk
    annual_inc                          SHAP=-0.2708  ↓ risk
    int_rate                            SHAP=-0.2628  ↓ risk


In [9]:
# ============================================================
# AGENT 2: DRIFT AGENT
# Role: Monitors the stream of predictions for signs of drift.
# Uses 3 sequential (online) detectors — they process loans
# one by one, unlike PSI which looks at batches.
#
#   ADWIN      — detects if the mean prediction is shifting
#   Page-Hinkley — detects a persistent upward shift in scores
#   DDM        — detects if error rate is increasing over time
#
# When drift is detected, Agent 2 raises an alert so Agent 3
# can trigger a human review (HITL).
# ============================================================

# ── Agent 2 FINAL — DDM with cooldown ──────────────────────

class DriftAgent:
    def __init__(self, window_size=500, adwin_threshold=0.03,
                 ph_threshold=50, ddm_drift=3.0, ddm_cooldown=500):

        # ADWIN
        self.adwin_window      = []
        self.adwin_window_size = window_size
        self.adwin_ref_mean    = None
        self.adwin_threshold   = adwin_threshold

        # Page-Hinkley
        self.ph_sum       = 0.0
        self.ph_min       = 0.0
        self.ph_threshold = ph_threshold
        self.ph_mean_ref  = None

        # DDM — with cooldown: after firing, wait 500 loans before next alert
        self.ddm_n           = 0
        self.ddm_errors      = 0
        self.ddm_min_rate    = float('inf')
        self.ddm_min_std     = float('inf')
        self.ddm_drift       = ddm_drift
        self.ddm_cooldown    = ddm_cooldown   # min loans between DDM alerts
        self.ddm_last_alert  = -ddm_cooldown  # allow firing immediately at start

        self.alerts = []
        self.total_idx = 0

    def update(self, score, true_label, idx):
        status = "OK"
        self.total_idx = idx

        # ── ADWIN ─────────────────────────────────────────────
        self.adwin_window.append(score)
        if len(self.adwin_window) > self.adwin_window_size:
            self.adwin_window.pop(0)
        if len(self.adwin_window) == self.adwin_window_size:
            win_mean = np.mean(self.adwin_window)
            if self.adwin_ref_mean is None:
                self.adwin_ref_mean = win_mean
            elif abs(win_mean - self.adwin_ref_mean) > self.adwin_threshold:
                status = "ADWIN_DRIFT"
                self.adwin_ref_mean = win_mean

        # ── Page-Hinkley ──────────────────────────────────────
        if self.ph_mean_ref is None:
            self.ph_mean_ref = score
        self.ph_sum += score - self.ph_mean_ref - 0.01
        self.ph_min  = min(self.ph_min, self.ph_sum)
        if (self.ph_sum - self.ph_min) > self.ph_threshold:
            status = "PH_DRIFT"
            self.ph_sum = 0.0; self.ph_min = 0.0
            self.ph_mean_ref = score

        # ── DDM with cooldown ─────────────────────────────────
        pred  = 1 if score >= 0.5 else 0
        error = int(pred != true_label)
        self.ddm_n += 1; self.ddm_errors += error
        if self.ddm_n > 200:  # need 200 samples to establish baseline
            rate = self.ddm_errors / self.ddm_n
            std  = np.sqrt(rate * (1 - rate) / self.ddm_n)
            if rate + std <= self.ddm_min_rate + self.ddm_min_std:
                self.ddm_min_rate = rate; self.ddm_min_std = std
            elif (rate + std > self.ddm_min_rate + self.ddm_drift * self.ddm_min_std
                  and idx - self.ddm_last_alert >= self.ddm_cooldown):
                status = "DDM_DRIFT"
                self.ddm_last_alert = idx   # start cooldown, do NOT reset counter

        if status != "OK":
            self.alerts.append({'idx': idx, 'type': status, 'score': round(score,4)})
        return status

# Re-run final Agent 2
agent2 = DriftAgent()
scores_5k = ensemble_predict(X_test.iloc[:5000])
for i, (score, label) in enumerate(zip(scores_5k, y_test[:5000])):
    agent2.update(score, label, i)

from collections import Counter
print(f"Drift alerts: {len(agent2.alerts)}")
for t, c in Counter(a['type'] for a in agent2.alerts).items():
    print(f"  {t}: {c}")
if agent2.alerts:
    for a in agent2.alerts:
        print(f"  Loan #{a['idx']:>5} | {a['type']:<15} | score={a['score']}")

Drift alerts: 10
  DDM_DRIFT: 9
  ADWIN_DRIFT: 1
  Loan #  524 | DDM_DRIFT       | score=0.094
  Loan # 1024 | DDM_DRIFT       | score=0.0248
  Loan # 1524 | DDM_DRIFT       | score=0.1089
  Loan # 2024 | DDM_DRIFT       | score=0.1191
  Loan # 2524 | DDM_DRIFT       | score=0.1102
  Loan # 2915 | ADWIN_DRIFT     | score=0.2419
  Loan # 3024 | DDM_DRIFT       | score=0.2767
  Loan # 3524 | DDM_DRIFT       | score=0.1426
  Loan # 4024 | DDM_DRIFT       | score=0.0571
  Loan # 4524 | DDM_DRIFT       | score=0.0658


In [11]:
# ============================================================
# AGENT 3: COMPLIANCE AGENT
# Role: Governance layer that sits above Agents 1 and 2.
#   1. Receives output from both agents
#   2. Decides if a loan needs Human-In-The-Loop (HITL) review
#   3. Writes every decision to an immutable audit trail
#      (required by SR 11-7, GDPR Article 22, EBA guidelines)
#
# HITL is triggered when:
#   - Agent 1 score is in the grey zone (0.40–0.60) — borderline
#   - Agent 2 has raised a drift alert at this loan index
#   - Score is extremely high (>= 0.75) — high-stakes decision
# ============================================================

import csv, os
from datetime import datetime

class ComplianceAgent:
    def __init__(self, audit_file='acrs_data/audit_trail.csv',
                 hitl_grey_low=0.40, hitl_grey_high=0.60, hitl_high_risk=0.75):

        self.hitl_grey_low  = hitl_grey_low
        self.hitl_grey_high = hitl_grey_high
        self.hitl_high_risk = hitl_high_risk
        self.audit_file     = audit_file

        # Create audit CSV with header (append mode so records accumulate)
        if not os.path.exists(audit_file):
            with open(audit_file, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow([
                    'timestamp', 'loan_idx', 'score', 'risk_label',
                    'drift_status', 'hitl_required', 'hitl_reason',
                    'top1_feature', 'top1_shap', 'decision'
                ])
        self.hitl_count  = 0
        self.total_count = 0

    def evaluate(self, loan_idx, agent1_result, drift_status):
        score      = agent1_result['score']
        label      = agent1_result['label']
        top1       = agent1_result['top3_shap'][0]   # most influential feature

        # ── HITL decision logic ───────────────────────────────
        hitl = False
        reason = "None"
        if self.hitl_grey_low <= score <= self.hitl_grey_high:
            hitl = True; reason = "Grey zone score (borderline)"
        elif drift_status != "OK":
            hitl = True; reason = f"Drift detected: {drift_status}"
        elif score >= self.hitl_high_risk:
            hitl = True; reason = "Extreme high-risk score"

        # ── Final decision ────────────────────────────────────
        # Auto-approve if low risk and no drift; flag for review otherwise
        if hitl:
            decision = "REFER_TO_HUMAN"
        elif score < 0.30:
            decision = "AUTO_APPROVE"
        else:
            decision = "AUTO_DECLINE"

        self.total_count += 1
        if hitl: self.hitl_count += 1

        # ── Write to audit trail ──────────────────────────────
        with open(self.audit_file, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([
                datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                loan_idx, score, label, drift_status,
                hitl, reason,
                top1[0], round(top1[1], 4), decision
            ])

        return {
            'decision':     decision,
            'hitl':         hitl,
            'hitl_reason':  reason
        }

agent3 = ComplianceAgent()
print("Agent 3 (Compliance) initialised")

# ── Full ACRS pipeline: all 3 agents together (100 loans) ───
print("\n--- ACRS Full Pipeline Demo (100 loans) ---")

# Reset Agent 2 for clean demo run
agent2_demo = DriftAgent()

summary = {'AUTO_APPROVE':0, 'AUTO_DECLINE':0, 'REFER_TO_HUMAN':0}

for i in range(100):
    loan       = X_test.iloc[[i]]
    a1_result  = agent1.score(loan)
    drift_stat = agent2_demo.update(a1_result['score'], y_test[i], i)
    a3_result  = agent3.evaluate(i, a1_result, drift_stat)
    summary[a3_result['decision']] += 1

print(f"\nDecision summary (100 loans):")
for k, v in summary.items():
    print(f"  {k:<20}: {v}")
print(f"\nHITL rate: {agent3.hitl_count}/{agent3.total_count} "
      f"= {agent3.hitl_count/agent3.total_count*100:.1f}%")
print(f"Audit trail saved: {agent3.audit_file}")

# Show last 3 audit rows
audit_df = pd.read_csv(agent3.audit_file)
print(f"\nAudit trail shape: {audit_df.shape}")
print(audit_df.tail(3).to_string(index=False))

Agent 3 (Compliance) initialised

--- ACRS Full Pipeline Demo (100 loans) ---

Decision summary (100 loans):
  AUTO_APPROVE        : 98
  AUTO_DECLINE        : 1
  REFER_TO_HUMAN      : 1

HITL rate: 1/100 = 1.0%
Audit trail saved: acrs_data/audit_trail.csv

Audit trail shape: (100, 10)
          timestamp  loan_idx  score  risk_label drift_status  hitl_required                  hitl_reason           top1_feature  top1_shap       decision
2026-09-19 22:27:39        97 0.0953    Low Risk           OK          False                          NaN             annual_inc    -0.2729   AUTO_APPROVE
2026-09-19 22:27:39        98 0.4531 Medium Risk           OK           True Grey zone score (borderline) purpose_small_business     0.3859 REFER_TO_HUMAN
2026-09-19 22:27:39        99 0.1579    Low Risk           OK          False                          NaN              loan_amnt    -0.3113   AUTO_APPROVE


In [13]:
# ── Layer 3 Final Metrics + Save ────────────────────────────
# Run full pipeline on all test loans to get final numbers
# (We score all, but only compute classification metrics)

from sklearn.metrics import roc_auc_score, f1_score
from scipy.stats import ks_2samp

print("Running full test set through ACRS pipeline...")
all_scores = ensemble_predict(X_test)   # all 504,194 loans

auc_l3  = roc_auc_score(y_test, all_scores)
ks_l3   = ks_2samp(all_scores[y_test==1], all_scores[y_test==0]).statistic
gini_l3 = 2 * auc_l3 - 1
f1_l3   = f1_score(y_test, (all_scores >= 0.5).astype(int))

print("\n" + "=" * 55)
print("LAYER 3 RESULTS SUMMARY")
print("=" * 55)
print(f"  Predictive Performance:")
print(f"    AUC-ROC : {auc_l3:.4f}")
print(f"    KS Stat : {ks_l3:.4f}")
print(f"    Gini    : {gini_l3:.4f}")
print(f"    F1 Score: {f1_l3:.4f}")
print(f"\n  Agentic Capability (new in Layer 3):")
print(f"    Agent 1 — Real-time SHAP per loan")
print(f"    Agent 2 — Drift: 10 alerts in 5000 loans (9 DDM, 1 ADWIN)")
print(f"    Agent 3 — HITL rate: 1.0% | Audit trail: 100 rows logged")
print(f"    Compliance: SR 11-7, GDPR Art.22, EBA guidelines")
print("=" * 55)

import json
l3_results = {
    "layer": 3,
    "auc":  round(auc_l3, 4),
    "ks":   round(ks_l3, 4),
    "gini": round(gini_l3, 4),
    "f1":   round(f1_l3, 4),
    "drift_alerts_5k": 10,
    "hitl_rate_pct": 1.0,
    "audit_rows": 100
}
with open('acrs_data/layer3_results.json', 'w') as f:
    json.dump(l3_results, f, indent=2)
print("Saved: layer3_results.json")

Running full test set through ACRS pipeline...

LAYER 3 RESULTS SUMMARY
  Predictive Performance:
    AUC-ROC : 0.7424
    KS Stat : 0.3433
    Gini    : 0.4848
    F1 Score: 0.2189

  Agentic Capability (new in Layer 3):
    Agent 1 — Real-time SHAP per loan
    Agent 2 — Drift: 10 alerts in 5000 loans (9 DDM, 1 ADWIN)
    Agent 3 — HITL rate: 1.0% | Audit trail: 100 rows logged
    Compliance: SR 11-7, GDPR Art.22, EBA guidelines
Saved: layer3_results.json
